In [2]:
import pandas as pd, numpy as np, json
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("demanda_paquetes_diaria.csv", parse_dates=["fecha"])
df = df.sort_values("fecha").reset_index(drop=True)
y = df["demanda_paquetes"]

print("Datos cargados y ordenados cronológicamente")
print(f"Filas: {len(df)} | Desde {df['fecha'].min().date()} hasta {df['fecha'].max().date()}")
print("Nulos por columna:")
print(df.isna().sum())
print("Variable objetivo (y): demanda_paquetes")

Datos cargados y ordenados cronológicamente
Filas: 180 | Desde 2026-01-05 hasta 2026-07-03
Nulos por columna:
fecha                 0
demanda_paquetes      0
campana_programada    0
dtype: int64
Variable objetivo (y): demanda_paquetes


In [3]:
df["dia_semana"] = df["fecha"].dt.dayofweek
df["mes"] = df["fecha"].dt.month
df["lag_1"] = y.shift(1)
df["lag_7"] = y.shift(7)
df["media_movil_7"] = y.shift(1).rolling(7).mean()
cols = ["dia_semana","mes","campana_programada","lag_1","lag_7","media_movil_7"]

print("Características creadas")
print("lag_1 y lag_7 usan solo demanda de días anteriores.")
print("media_movil_7 usa shift(1): excluye la demanda del mismo día (evita fuga de datos).")
print(df[["fecha","demanda_paquetes","lag_1","lag_7","media_movil_7"]].head(10))

Características creadas
lag_1 y lag_7 usan solo demanda de días anteriores.
media_movil_7 usa shift(1): excluye la demanda del mismo día (evita fuga de datos).
       fecha  demanda_paquetes    lag_1   lag_7  media_movil_7
0 2026-01-05              8962      NaN     NaN            NaN
1 2026-01-06              8544   8962.0     NaN            NaN
2 2026-01-07              8577   8544.0     NaN            NaN
3 2026-01-08             10146   8577.0     NaN            NaN
4 2026-01-09             10383  10146.0     NaN            NaN
5 2026-01-10              8981  10383.0     NaN            NaN
6 2026-01-11              7854   8981.0     NaN            NaN
7 2026-01-12              9178   7854.0  8962.0    9063.857143
8 2026-01-13              8869   9178.0  8544.0    9094.714286
9 2026-01-14              8553   8869.0  8577.0    9141.142857


In [4]:
filas_antes = len(df)
d = df.dropna(subset=cols).reset_index(drop=True)
corte = int(len(d) * 0.8)
train, test = d.iloc[:corte], d.iloc[corte:]
X_train, y_train = train[cols], train["demanda_paquetes"]
X_test,  y_test  = test[cols],  test["demanda_paquetes"]

print("Limpieza y división cronológica 80/20 (sin mezclar)")
print(f"Filas eliminadas por falta de historial: {filas_antes - len(d)}")
print(f"Train: {len(train)} filas ({train['fecha'].min().date()} a {train['fecha'].max().date()})")
print(f"Test : {len(test)} filas ({test['fecha'].min().date()} a {test['fecha'].max().date()})")
print("El entrenamiento es anterior a la prueba:", train["fecha"].max() < test["fecha"].min())

Limpieza y división cronológica 80/20 (sin mezclar)
Filas eliminadas por falta de historial: 7
Train: 138 filas (2026-01-12 a 2026-05-29)
Test : 35 filas (2026-05-30 a 2026-07-03)
El entrenamiento es anterior a la prueba: True


In [5]:
base = DummyRegressor(strategy="mean").fit(X_train, y_train)
modelo = LinearRegression().fit(X_train, y_train)
p_base = base.predict(X_test)
p_mod = modelo.predict(X_test)

print("Modelos entrenados solo con datos de train")
print(f"Baseline (predice siempre el promedio): {p_base[0]:.1f} paquetes")
print("Coeficientes de LinearRegression:")
for c, v in zip(cols, modelo.coef_):
    print(f"  {c}: {v:.2f}")

Modelos entrenados solo con datos de train
Baseline (predice siempre el promedio): 9304.2 paquetes
Coeficientes de LinearRegression:
  dia_semana: -111.19
  mes: 128.36
  campana_programada: 1314.67
  lag_1: 0.13
  lag_7: 0.58
  media_movil_7: -0.24


In [6]:
mae_b = mean_absolute_error(y_test, p_base)
mae_m = mean_absolute_error(y_test, p_mod)
rmse_m = float(np.sqrt(mean_squared_error(y_test, p_mod)))
r2 = r2_score(y_test, p_mod)
mejora = (mae_b - mae_m) / mae_b * 100

print("Métricas sobre el conjunto de prueba")
print(f"MAE baseline: {mae_b:.2f} paquetes")
print(f"MAE modelo  : {mae_m:.2f} paquetes")
print(f"RMSE modelo : {rmse_m:.2f}")
print(f"R²          : {r2:.4f}")
print(f"Mejora del MAE: {mejora:.2f}%")

Métricas sobre el conjunto de prueba
MAE baseline: 1001.34 paquetes
MAE modelo  : 340.13 paquetes
RMSE modelo : 413.16
R²          : 0.8256
Mejora del MAE: 66.03%


In [7]:
pd.DataFrame({"fecha": test["fecha"].dt.strftime("%Y-%m-%d"),
              "real": y_test.values, "baseline": p_base,
              "prediccion": p_mod}).to_csv("predicciones.csv", index=False)

metricas = {"mae_baseline": float(mae_b), "mae_modelo": float(mae_m),
            "rmse_modelo": rmse_m, "r2": float(r2), "mejora_mae_pct": float(mejora)}
json.dump(metricas, open("metricas.json","w"), indent=2)

print("Archivos exportados")
print("predicciones.csv (orden temporal):")
print(pd.read_csv("predicciones.csv").head())
print("metricas.json:", metricas)

Archivos exportados
predicciones.csv (orden temporal):
        fecha   real     baseline    prediccion
0  2026-05-30   8754  9304.181159   8676.055702
1  2026-05-31   7790  9304.181159   8321.886883
2  2026-06-01  10272  9304.181159  10550.417783
3  2026-06-02   9922  9304.181159  10790.713325
4  2026-06-03   9878  9304.181159  10606.498847
metricas.json: {'mae_baseline': 1001.3405797101449, 'mae_modelo': 340.13416958444725, 'rmse_modelo': 413.1612420518522, 'r2': 0.8255633828948655, 'mejora_mae_pct': 66.03211969269188}


In [8]:
print(f"Prueba: {test['fecha'].min().date()} a {test['fecha'].max().date()}. MAE baseline {mae_b:.2f}, MAE modelo {mae_m:.2f}, mejora {mejora:.2f}%.")
print("El modelo SÍ agrega valor frente al baseline." if mejora > 0 else "El modelo NO agrega valor frente al baseline.")
print("No debe entrenarse con el futuro: causaría fuga temporal y métricas infladas, y en producción ese dato no existe.")
print("Limitan el uso real: modelo lineal simple, prueba de solo 35 días y dependencia de la campaña programada y de la demanda previa.")
print("Es una estimación inicial, no una certeza.")

Prueba: 2026-05-30 a 2026-07-03. MAE baseline 1001.34, MAE modelo 340.13, mejora 66.03%.
El modelo SÍ agrega valor frente al baseline.
No debe entrenarse con el futuro: causaría fuga temporal y métricas infladas, y en producción ese dato no existe.
Limitan el uso real: modelo lineal simple, prueba de solo 35 días y dependencia de la campaña programada y de la demanda previa.
Es una estimación inicial, no una certeza.
